In [18]:
"""
PICASO Multiprocessing Grid Runner
Runs PICASO atmosphere models in parallel across a parameter grid
"""

import os
import sys

# Set environment variables FIRST before any other imports
picaso_refdata = "/Users/nguyendat/Documents/GitHub/picaso/reference/"
pysyn_cdbs = "/Users/nguyendat/Documents/GitHub/picaso/reference/stellar_spectra/grp/redcat/trds"
picaso_home = '/Users/nguyendat/Documents/GitHub/picaso/'

os.environ['picaso_refdata'] = picaso_refdata
os.environ['PYSYN_CDBS'] = pysyn_cdbs

# Database paths
sonora_profile_db = '/Users/nguyendat/Documents/GitHub/picaso/data/sonora_profile/'
virga_directory = '/Users/nguyendat/Documents/GitHub/picaso/data/virga/'

# Force reload of modified module
import importlib
import virga.justdoit as vjdi
import virga.justplotit as vjpi
import picaso.justdoit as jdi
import picaso.justplotit as jpi

importlib.reload(vjdi)

# Now import everything else
import numpy as np
import pandas as pd
import pickle
from multiprocessing import Pool, cpu_count
from itertools import product
from datetime import datetime
import astropy.units as u


In [36]:
# ============================================================================
def configure_atm(C_to_O=0.55, mh=1.0, Teff=1200, gravity=10000, 
                  kzz=1e7, phase=0, wave_range=[1.0,6.0], eq=False, excluded_mol=None):
    """Configure brown dwarf atmosphere."""
    opa = jdi.opannection(wave_range=wave_range)
    bd = jdi.inputs(calculation='browndwarf')
    bd.phase_angle(phase)
    bd.gravity(gravity=gravity, gravity_unit=u.Unit('m/(s**2)'))
    bd.sonora(sonora_profile_db, Teff)
    
    if eq:
        bd.chemeq_visscher(C_to_O, mh)
    
    profile = bd.inputs['atmosphere']['profile']
    profile['kzz'] = np.ones(profile['pressure'].shape[0]) * kzz
    profile['kz'] = profile['kzz'].copy()
    bd.atmosphere(df=profile, exclude_mol=excluded_mol)
    
    return bd, opa, profile

# ===========================================================================
def convert_and_regrid(df, R=500):
    """Convert spectrum to different units and regrid."""
    x, y = df['wavenumber'], df['thermal']
    xmicron = 1e4/x
    flamy = y*1e-8
    sp = jdi.psyn.ArraySpectrum(xmicron, flamy, waveunits='um', fluxunits='FLAM')
    sp.convert("um")
    sp.convert('Fnu')
    
    x = sp.wave
    y = sp.flux
    df['fluxnu'] = y
    x, y = jdi.mean_regrid(x, y, R=R)
    df['regridy'] = y
    df['regridx'] = x
    
    return df

# ===========================================================================
def configure_cloud(df, profile, opacity, mh=1, mmw=2.2, fsed=1, R=300,
                    gases=['Fe', 'MgSiO3', 'Mg2SiO4', 'Al2O3']):
    """Configure clouds in atmosphere."""
    virga_available_condensates = ['ZnS', 'TiO2', 'NH3', 'Na2S', 'MnS', 
                                   'MgSiO3', 'Mg2SiO4', 'KCl', 'H2O', 
                                   'Fe', 'Cr', 'CH4', 'CaTiO3', 'Al2O3']
    
    if gases == 'recommended':
        recommended_gases = vjdi.recommend_gas(profile['pressure'], 
                                              profile['temperature'], 
                                              mh=mh, mmw=mmw, plot=False)
        gases = [g for g in recommended_gases if g in virga_available_condensates]
        print(f"Using recommended gases: {gases}")
    else:
        gases = [g for g in gases if g in virga_available_condensates]
        print(f"Using specified gases (Virga-compatible only): {gases}")
        if len(gases) == 0:
            print("WARNING: No valid condensate gases specified for Virga. No clouds will be added.")
    
    cld_out = df.virga(gases, virga_directory, fsed=fsed, mmw=mmw)
    df_out = df.spectrum(opacity, full_output=True)
    df_out = convert_and_regrid(df_out, R=R)
    
    return df_out['regridx'], df_out['regridy'], cld_out, df_out

In [16]:
# Test your changes
gases = ['Fe', 'MgSiO3']

# Test 1: Float fsed (backward compatibility)
try:
    atm = vjdi.Atmosphere(gases, fsed=1.0, mh=1.0, mmw=2.2)
    print("✓ Float fsed works")
except Exception as e:
    print(f"✗ Float fsed failed: {e}")

# Test 2: Dict fsed (new feature)
try:
    atm = vjdi.Atmosphere(gases, fsed={'Fe': 1.0, 'MgSiO3': 1.5}, mh=1.0, mmw=2.2)
    print("✓ Dict fsed works")
except Exception as e:
    print(f"✗ Dict fsed failed: {e}")

# Test 3: Validation
try:
    atm = vjdi.Atmosphere(gases, fsed={'Fe': 1.0})  # Missing MgSiO3
    print("✗ Validation should have failed!")
except ValueError as e:
    print(f"✓ Validation works: {e}")


✓ Float fsed works
✓ Dict fsed works
✓ Validation works: fsed dict is missing the following species from condensibles: {'MgSiO3'}
All species in condensibles must have fsed values specified.


In [ ]:
bd, opa, profile = configure_atm(C_to_O=0.55, mh=1.0, Teff=1200, gravity=10000,
                                 kzz=1e7, phase=0, wave_range=[1.0,6.0], eq=False)

In [38]:
"""Ensure no performance regression with single fsed"""
gases = ['Fe', 'MgSiO3', 'Mg2SiO4', 'Al2O3']
fsed = 1.0
# Time original behavior (float input)

start1 = time.perf_counter()

x1, y1, cld_out1, df_out2 = configure_cloud(
    bd, profile, opa,
    mh=1.0, mmw=2.2, fsed=fsed, R=300,
    gases=gases)

elapsed1 = time.perf_counter() - start1
print(f"Single fsed time: {elapsed1:.3f} seconds")


"""Benchmark dict input with equivalent values"""
fsed_dict={'Fe': 1.0, 'MgSiO3': 1.0, 'Mg2SiO4': 1.0, 'Al2O3': 1.0}

start2 = time.perf_counter()

# Dict with same values
x2, y2, cld_out2, df_out2 = configure_cloud(
    bd, profile, opa,
    mh=1.0, mmw=2.2, fsed=fsed_dict, R=300,
    gases=gases)

elapsed2 = time.perf_counter() - start2

print(f"Dict fsed time: {elapsed2:.3f} seconds")

# Should be within 1% of each other
print(f"Performance time-difference: {(abs(time_dict - time_single) / time_single) * 100:.2f}%")

Using specified gases (Virga-compatible only): ['Fe', 'MgSiO3', 'Mg2SiO4', 'Al2O3']
Take caution in analyzing results. There have been a calculated particle radii off the Mie grid, which has a min radius of 1e-08cm and distribution of 2. The following errors:7.250668959330075e-08cm for the 0th gas at the 0th grid point; 1.7818376472825065e-07cm for the 1th gas at the 0th grid point; 1.7696241363287198e-07cm for the 2th gas at the 0th grid point; 1.426079664446073e-07cm for the 3th gas at the 0th grid point; 8.398708737489388e-08cm for the 0th gas at the 1th grid point; 2.0633555660913758e-07cm for the 1th gas at the 1th grid point; 2.0492184533269938e-07cm for the 2th gas at the 1th grid point; 1.651543004600049e-07cm for the 3th gas at the 1th grid point; 9.720731055378375e-08cm for the 0th gas at the 2th grid point; 2.3874315098881316e-07cm for the 1th gas at the 2th grid point; 2.3710811655525808e-07cm for the 2th gas at the 2th grid point; 1.9111204709948305e-07cm for the 3th gas a

In [39]:
"""The Spectrum should be identical"""

from bokeh.plotting import show, figure
from bokeh.io import output_notebook 
output_notebook()

show(jpi.spectrum([x1, x2], [y1, y2], 
                plot_width=1000, plot_height=500,
                y_axis_label='Flux (erg/s/cm^2/Hz)',
                x_axis_label='Wavelength (micron)',
                y_axis_type='log',
                title='Single vs Dict fsed Spectrum Comparison',
                legend=['single-fsed', 'dict-fsed']))

if np.allclose(y1, y2):
    print("✓ Spectra are identical between single and dict fsed inputs.")

Loading BokehJS ...

✓ Spectra are identical between single and dict fsed inputs.


In [31]:
"""Now let's try varying fsed per species"""

gases = ['Fe', 'Al2O3', 'MgSiO3', 'Mg2SiO4']
fsed_dict0 = {'Fe': 1.0, 'Al2O3': 1.0, 'MgSiO3': 1.0, 'Mg2SiO4': 1.0}
fsed_dict1 = {'Fe': 1.0, 'Al2O3': 1.0, 'MgSiO3': 1.05, 'Mg2SiO4': 1.05}
fsed_dict2 = {'Fe': 1.0, 'Al2O3': 1.0, 'MgSiO3': 1.10, 'Mg2SiO4': 1.10}

start2 = time.perf_counter()

# Dict with same values
x0, y0, cld_out0, df_out0 = configure_cloud(
    bd, profile, opa,
    mh=1.0, mmw=2.2, fsed=fsed_dict0, R=300,
    gases=gases)

x1, y1, cld_out1, df_out1 = configure_cloud(
    bd, profile, opa,
    mh=1.0, mmw=2.2, fsed=fsed_dict1, R=300,
    gases=gases)

x2, y2, cld_out2, df_out2 = configure_cloud(
    bd, profile, opa,
    mh=1.0, mmw=2.2, fsed=fsed_dict2, R=300,
    gases=gases)


Using specified gases (Virga-compatible only): ['Fe', 'Al2O3', 'MgSiO3', 'Mg2SiO4']
Take caution in analyzing results. There have been a calculated particle radii off the Mie grid, which has a min radius of 1e-08cm and distribution of 2. The following errors:7.250668959330075e-08cm for the 0th gas at the 0th grid point; 1.426079664446073e-07cm for the 1th gas at the 0th grid point; 1.7818376472825065e-07cm for the 2th gas at the 0th grid point; 1.7696241363287198e-07cm for the 3th gas at the 0th grid point; 8.398708737489388e-08cm for the 0th gas at the 1th grid point; 1.651543004600049e-07cm for the 1th gas at the 1th grid point; 2.0633555660913758e-07cm for the 2th gas at the 1th grid point; 2.0492184533269938e-07cm for the 3th gas at the 1th grid point; 9.720731055378375e-08cm for the 0th gas at the 2th grid point; 1.9111204709948305e-07cm for the 1th gas at the 2th grid point; 2.3874315098881316e-07cm for the 2th gas at the 2th grid point; 2.3710811655525808e-07cm for the 3th gas a

In [ ]:
show(jpi.spectrum([x0, x1, x2], [y0, y1, y2], 
                plot_width=1000, plot_height=500,
                y_axis_label='Flux (erg/s/cm^2/Hz)',
                x_axis_label='Wavelength (micron)',
                y_axis_type='log',
                title='Fsed per-species Comparison: Fe, Al2O3=1.0, Only silicates are changing',
                legend=['silicates = 1.0', 'silicates = 1.05', 'silicates = 1.10']))

In [62]:
"""Now let's try varying fsed per species such that species are anti correlated"""

gases = ['Fe', 'Al2O3', 'MgSiO3', 'Mg2SiO4']
a0, b0 = 1.10, 1.00
a1, b1 = 1.05, 1.15
a2, b2 = 1.0, 1.30

fsed_dict0 = {'Fe': a0, 'Al2O3': a0, 'MgSiO3': b0, 'Mg2SiO4': b0}
fsed_dict1 = {'Fe': a1, 'Al2O3': a1, 'MgSiO3': b1, 'Mg2SiO4': b1}
fsed_dict2 = {'Fe': a2, 'Al2O3': a2, 'MgSiO3': b2, 'Mg2SiO4': b2}

start2 = time.perf_counter()

# Dict with same values
x0, y0, cld_out0, df_out0 = configure_cloud(
    bd, profile, opa,
    mh=1.0, mmw=2.2, fsed=fsed_dict0, R=500,
    gases=gases)

x1, y1, cld_out1, df_out1 = configure_cloud(
    bd, profile, opa,
    mh=1.0, mmw=2.2, fsed=fsed_dict1, R=500,
    gases=gases)

x2, y2, cld_out2, df_out2 = configure_cloud(
    bd, profile, opa,
    mh=1.0, mmw=2.2, fsed=fsed_dict2, R=500,
    gases=gases)


Using specified gases (Virga-compatible only): ['Fe', 'Al2O3', 'MgSiO3', 'Mg2SiO4']
Take caution in analyzing results. There have been a calculated particle radii off the Mie grid, which has a min radius of 1e-08cm and distribution of 2. The following errors:8.576961961023328e-08cm for the 0th gas at the 0th grid point; 1.6810747175106303e-07cm for the 1th gas at the 0th grid point; 1.7818376472825065e-07cm for the 2th gas at the 0th grid point; 1.7696241363287198e-07cm for the 3th gas at the 0th grid point; 9.925959022921102e-08cm for the 0th gas at the 1th grid point; 1.944839953536877e-07cm for the 1th gas at the 1th grid point; 2.0633555660913758e-07cm for the 2th gas at the 1th grid point; 2.0492184533269938e-07cm for the 3th gas at the 1th grid point; 1.1477450896753673e-07cm for the 0th gas at the 2th grid point; 2.248109012538259e-07cm for the 1th gas at the 2th grid point; 2.3874315098881316e-07cm for the 2th gas at the 2th grid point; 2.3710811655525808e-07cm for the 3th gas 

In [63]:
show(jpi.spectrum([x0, x1, x2], [y0, y1, y2], 
                plot_width=1000, plot_height=500,
                y_axis_label='Flux (erg/s/cm^2/Hz)',
                x_axis_label='Wavelength (micron)',
                y_axis_type='log',
                title='Fsed per-species Comparison: Fe, Al2O3 is anti-correlated with silicates',
                legend=[f'fe={a0}, si={b0}', 
                        f'fe={a1}, si={b1}', 
                        f'fe={a2}, si={b2}']))

In [ ]:
"""Adding Na2S to the mix"""

gases = ['Fe', 'Al2O3', 'MgSiO3', 'Mg2SiO4', 'Na2S']
a0, b0 = 1.15, 1.00
a1, b1 = 1.0, 1.15
a2, b2 = 0.85, 1.30

fsed_dict0 = {'Fe': a0, 'Al2O3': a0, 'MgSiO3': a0, 'Mg2SiO4': a0, 'Na2S': b0}
fsed_dict1 = {'Fe': a1, 'Al2O3': a1, 'MgSiO3': a1, 'Mg2SiO4': a1, 'Na2S': b1}
fsed_dict2 = {'Fe': a2, 'Al2O3': a2, 'MgSiO3': a2, 'Mg2SiO4': a2, 'Na2S': b2}

start2 = time.perf_counter()

# Dict with same values
x0, y0, cld_out0, df_out0 = configure_cloud(
    bd, profile, opa, fsed=fsed_dict0, R=500,
    gases=gases)

x1, y1, cld_out1, df_out1 = configure_cloud(
    bd, profile, opa, fsed=fsed_dict1, R=500,
    gases=gases)

x2, y2, cld_out2, df_out2 = configure_cloud(
    bd, profile, opa, fsed=fsed_dict2, R=500,
    gases=gases)

Using specified gases (Virga-compatible only): ['Fe', 'Al2O3', 'MgSiO3', 'Mg2SiO4', 'Na2S']
Take caution in analyzing results. There have been a calculated particle radii off the Mie grid, which has a min radius of 1e-08cm and distribution of 2. The following errors:8.576961961023328e-08cm for the 0th gas at the 0th grid point; 1.6810747175106303e-07cm for the 1th gas at the 0th grid point; 2.1009986166583837e-07cm for the 2th gas at the 0th grid point; 2.0865813797051397e-07cm for the 3th gas at the 0th grid point; 3.054297291241391e-07cm for the 4th gas at the 0th grid point; 9.925959022921102e-08cm for the 0th gas at the 1th grid point; 1.944839953536877e-07cm for the 1th gas at the 1th grid point; 2.4302400884146836e-07cm for the 2th gas at the 1th grid point; 2.4135765832053396e-07cm for the 3th gas at the 1th grid point; 1.1477450896753673e-07cm for the 0th gas at the 2th grid point; 2.248109012538259e-07cm for the 1th gas at the 2th grid point; 2.808717333355851e-07cm for the 2t

In [ ]:
show(jpi.spectrum([x0, x1, x2], [y0, y1, y2], 
                plot_width=1000, plot_height=500,
                y_axis_label='Flux (erg/s/cm^2/Hz)',
                x_axis_label='Wavelength (micron)',
                y_axis_type='log',
                title='Fsed per-species Comparison: Na2S is anti-correlated with the rest',
                legend=[f'fe,si={a0}, na={b0}', 
                        f'fe,si={a1}, na={b1}', 
                        f'fe,si={a2}, na={b2}']))